# Pass-Rusher Gravity GNN — Pipeline Notebook (GNN-V1)

End-to-end run for the pass-rusher gravity Graph Neural Network. The plan is described in detail in the branch README (`Data Cleaning + Engineering/gnn/README.md`). This notebook is the canonical entrypoint; all heavy lifting lives in importable Python modules under `gnn/`.

Pipeline stages:
1. Load `play_context.csv`, `play_attention_scores.csv`, and player metadata.
2. Filter to gravity-eligible pass-rush plays.
3. Stream `gravity_base.csv` and build per-frame PyTorch-Geometric graphs across the pass-rush window.
4. Run two tabular baselines (Ridge, HistGradientBoosting) on the same labels for comparison.
5. Train GraphSAGE and GATv2 expected-attention models with grouped (gameId) train/val/test split.
6. Aggregate frame predictions to play-rusher level, write `gnn_attention_scores.csv` and `gnn_player_gravity.csv`.
7. Validate (per-position metrics, split-half stability, top players).

In [1]:
import sys; print(sys.executable)

import os 
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['OMP_NUM_THREADS'] = '1'

/opt/anaconda3/bin/python


## 0. Environment setup
If you haven't already, install dependencies via `pip install -r requirements.txt` or `conda env create -f environment.yml`. PyTorch Geometric requires PyTorch be installed first; see the README for platform-specific install commands.

In [ ]:
import sys
from pathlib import Path

# Make `gnn` importable regardless of where the notebook is launched from.
NOTEBOOK_DIR = Path.cwd()
DCE_DIR = NOTEBOOK_DIR if (NOTEBOOK_DIR / 'gnn').exists() else NOTEBOOK_DIR.parent
if str(DCE_DIR) not in sys.path:
    sys.path.insert(0, str(DCE_DIR))
print('importing gnn from', DCE_DIR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from gnn import config
from gnn.config import TrainConfig, pick_device
from gnn.dataset import build_frame_graphs, save_frame_graphs, load_frame_graphs, to_pyg_dataset, cache_path
from gnn.data_filters import filter_eligible_plays
from gnn.splits import grouped_split
from gnn.models import build_model
from gnn.train import train_model
from gnn.evaluate import predict_play_rusher, regression_metrics, metrics_by_position_group, split_half_stability
from gnn.infer import write_play_rusher_predictions, write_player_gravity
from gnn.lookups import rusher_position_group_lookup
from gnn.baselines import build_baseline_table, fit_linear_baseline, fit_tree_baseline

# Knobs you'll usually touch: max_plays caps the eligible play count for fast iteration; set to None for full run.
MAX_PLAYS = 400  # set to None to use all eligible plays
FRAMES_PER_PLAY = config.FRAMES_PER_PLAY
# Auto-detect: cuda > mps (Apple Silicon) > cpu. Override with 'cuda', 'mps', or 'cpu' if you want to force one.
DEVICE = pick_device('auto')
RANDOM_SEED = 42
print('config:', dict(max_plays=MAX_PLAYS, frames_per_play=FRAMES_PER_PLAY, device=DEVICE))
print('torch:', torch.__version__, '| cuda available:', torch.cuda.is_available(),
      '| mps available:', getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available())

## 1. Load play context and labels

In [3]:
play_context = pd.read_csv(config.PLAY_CONTEXT_CSV)
play_attention = pd.read_csv(config.PLAY_ATTENTION_CSV)
for c in ['gameId', 'playId']:
    play_context[c] = play_context[c].astype(int)
    play_attention[c] = play_attention[c].astype(int)
play_attention['rusher_nflId'] = play_attention['rusher_nflId'].astype(int)

print('plays in context:', len(play_context))
print('rusher rows in attention:', len(play_attention))
play_attention.head()

plays in context: 8557
rusher rows in attention: 30971


,gameId,playId,rusher_nflId,avg_attention_score,rusher_name
0,2021090900,97,41263,0.461538,Demarcus Lawrence
1,2021090900,97,42403,0.615385,Randy Gregory
2,2021090900,97,44955,1.307692,Carlos Watkins
3,2021090900,97,53441,1.000000,Micah Parsons
4,2021090900,97,53504,0.615385,Osa Odighizuwa


In [4]:
eligible = filter_eligible_plays(play_context)
print(f'eligible plays after filter: {len(eligible):,}  (of {len(play_context):,})')
eligible[['offenseFormation','passCoverage','passCoverageType','dropBackType','playAction']].describe(include='all').T

eligible plays after filter: 7,441  (of 8,557)


,count,unique,top,freq
offenseFormation,7441,7,SHOTGUN,4988
passCoverage,7441,12,Cover 3,2288
passCoverageType,7441,3,Zone,4917
dropBackType,7441,2,Traditional,6542
playAction,7441,2,False,6045


## 2. Build frame-level graphs

Cached to `Data Cleaning + Engineering/gnn/cache/`. Re-run with `force=True` if the schema or filters change.

In [5]:
cache_file = cache_path(f'frame_graphs_max{MAX_PLAYS}_fp{FRAMES_PER_PLAY}.pkl')
force_rebuild = False
if cache_file.exists() and not force_rebuild:
    print('loading cached graphs from', cache_file)
    graphs = load_frame_graphs(cache_file)
else:
    graphs = build_frame_graphs(
        play_context=play_context,
        play_attention=play_attention,
        frames_per_play=FRAMES_PER_PLAY,
        max_plays=MAX_PLAYS,
        chunksize=300_000,
        verbose=True,
    )
    save_frame_graphs(graphs, cache_file)
print(f'built {len(graphs)} frame graphs')

loading cached graphs from /Users/moulikchatterjee/Documents/UCLA /Clubs and Activities/Bruin Sports Analytics/Research/Football Research 2026/Football Gravity Metrics/BSA-Football-2026-Gravity-Metrics/gnn/cache/frame_graphs_max400_fp6.pkl
built 2400 frame graphs


In [6]:
# Quick sanity look at the first graph
g0 = graphs[0]
print('nodes:', g0.x.shape, '\nedge_index:', g0.edge_index.shape, '\nedge_attr:', g0.edge_attr.shape)
print('rushers in frame:', int(g0.rusher_mask.sum()))
print('targets:', g0.y[g0.rusher_mask])
print('global feature dim:', g0.u.shape)

nodes: (22, 40) 
edge_index: (2, 462) 
edge_attr: (462, 9)
rushers in frame: 4
targets: [1.   0.96 0.52 2.04]
global feature dim: (42,)


## 3. Convert to PyTorch-Geometric and split by gameId
Holding out by `gameId` so train/val/test never share frames or plays.

In [7]:
data_list = to_pyg_dataset(graphs)
game_ids = [int(g.game_id) for g in graphs]
split = grouped_split(game_ids, val_fraction=0.15, test_fraction=0.15, seed=RANDOM_SEED)
train_data = [data_list[i] for i in split.train_idx]
val_data = [data_list[i] for i in split.val_idx]
test_data = [data_list[i] for i in split.test_idx]
print('train graphs', len(train_data), 'val', len(val_data), 'test', len(test_data))
print('train games', len(split.train_games), 'val', len(split.val_games), 'test', len(split.test_games))

train graphs 1596 val 396 test 408
train games 82 val 18 test 18


## 4. Tabular baselines

We compare the GNN against ridge regression and a HistGradientBoosting tree on the same labels (`avg_attention_score`) and play-level globals. The GNN must beat or match these on held-out MAE/RMSE for the graph structure to be earning its keep.

In [8]:
rusher_pos_lookup = rusher_position_group_lookup()
tabular = build_baseline_table(play_attention, play_context, rusher_pos_lookup)
tabular_train = tabular[tabular['gameId'].isin(split.train_games)].reset_index(drop=True)
tabular_val = tabular[tabular['gameId'].isin(split.val_games)].reset_index(drop=True)
tabular_test = tabular[tabular['gameId'].isin(split.test_games)].reset_index(drop=True)
print('tabular sizes train/val/test:', len(tabular_train), len(tabular_val), len(tabular_test))

tabular sizes train/val/test: 21164 4458 4552


In [9]:
ridge_model, _, ridge_val_pred = fit_linear_baseline(tabular_train, tabular_val)
tree_model, _, tree_val_pred = fit_tree_baseline(tabular_train, tabular_val)

ridge_metrics = regression_metrics(tabular_val['avg_attention_score'].to_numpy(), ridge_val_pred)
tree_metrics = regression_metrics(tabular_val['avg_attention_score'].to_numpy(), tree_val_pred)
pd.DataFrame({'ridge': ridge_metrics, 'tree': tree_metrics}).T

,mae,rmse,r2,spearman,pearson,n
ridge,0.331123,0.424335,0.246479,0.482994,0.497624,4458.0
tree,0.328854,0.421725,0.255718,0.494371,0.508249,4458.0


## 5. Train GraphSAGE

In [10]:
cfg = TrainConfig(epochs=20, batch_size=128, lr=1e-3, hidden_dim=64, num_layers=3, dropout=0.2,
                  device=DEVICE, seed=RANDOM_SEED, early_stopping_patience=5)
sage_model = build_model('graphsage', hidden_dim=cfg.hidden_dim, num_layers=cfg.num_layers, dropout=cfg.dropout)
sage_history = train_model(sage_model, train_data, val_data, cfg=cfg, checkpoint_name='graphsage_best.pt')
print('GraphSAGE best val MSE:', sage_history.best_val_loss)

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(sage_history.train_losses, label='train')
ax.plot(sage_history.val_losses, label='val')
ax.set_xlabel('epoch'); ax.set_ylabel('masked MSE'); ax.set_title('GraphSAGE training'); ax.legend(); plt.tight_layout()

## 6. Train GATv2

In [ ]:
gat_model = build_model('gatv2', hidden_dim=cfg.hidden_dim, num_layers=cfg.num_layers, dropout=cfg.dropout, heads=4, use_edge_features=True)
gat_history = train_model(gat_model, train_data, val_data, cfg=cfg, checkpoint_name='gatv2_best.pt')
print('GATv2 best val MSE:', gat_history.best_val_loss)

## 7. Evaluate on the held-out test split

In [ ]:
sage_pred_df = predict_play_rusher(sage_model, test_data, device=DEVICE)
gat_pred_df = predict_play_rusher(gat_model, test_data, device=DEVICE)

sage_metrics = regression_metrics(sage_pred_df['actual_attention_gnn'].to_numpy(), sage_pred_df['expected_attention_gnn'].to_numpy())
gat_metrics = regression_metrics(gat_pred_df['actual_attention_gnn'].to_numpy(), gat_pred_df['expected_attention_gnn'].to_numpy())

summary = pd.DataFrame({
    'ridge_val': ridge_metrics,
    'tree_val': tree_metrics,
    'graphsage_test': sage_metrics,
    'gatv2_test': gat_metrics,
}).T
summary

In [ ]:
metrics_by_position_group(gat_pred_df, rusher_pos_lookup)

## 8. Choose the better model and write outputs

Pick the model with the lower test MAE. Write `gnn_attention_scores.csv` (play-rusher) and `gnn_player_gravity.csv` (player rollup, z-scored).

In [ ]:
best_name, best_pred_df = ('gatv2', gat_pred_df) if gat_metrics['mae'] <= sage_metrics['mae'] else ('graphsage', sage_pred_df)
print('selected', best_name)

# Run inference on ALL graphs so the output covers every eligible play, not just the test split.
best_model = gat_model if best_name == 'gatv2' else sage_model
all_pred_df = predict_play_rusher(best_model, data_list, device=DEVICE)

play_rusher = write_play_rusher_predictions(all_pred_df, play_attention)
player_df = write_player_gravity(play_rusher, rusher_position_lookup=rusher_pos_lookup, min_plays=25)
print('wrote', len(play_rusher), 'play-rusher rows to', config.GNN_ATTENTION_CSV)
print('wrote', len(player_df), 'qualified players to', config.GNN_PLAYER_GRAVITY_CSV)
player_df.head(15)

## 9. Validation
Three quick checks aligned with the literature:
- **Split-half stability** — gravity_gnn correlates across random halves of plays per player.
- **Position group breakdown** — Edge and DI groups should not be systematically negative.
- **Top performers** — pass the eye test against known elite rushers.

In [ ]:
stab = split_half_stability(play_rusher, seed=RANDOM_SEED)
print('split-half Spearman of gravity_gnn:', round(stab, 3) if stab == stab else 'nan')

if 'position_group' in player_df.columns:
    print(player_df.groupby('position_group')['mean_gravity'].describe())

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
ax.scatter(player_df['mean_expected'], player_df['mean_actual'], alpha=0.5)
lim = (min(player_df['mean_expected'].min(), player_df['mean_actual'].min()) - 0.05,
       max(player_df['mean_expected'].max(), player_df['mean_actual'].max()) + 0.05)
ax.plot(lim, lim, 'k--', lw=1)
ax.set_xlabel('Expected attention (GNN)')
ax.set_ylabel('Actual attention')
ax.set_title('Player-level actual vs expected (above line = positive gravity)')
for _, r in player_df.head(8).iterrows():
    ax.annotate(r['rusher_name'], (r['mean_expected'], r['mean_actual']), fontsize=8)
plt.tight_layout()

In [ ]:
print('Top 15 by gravity_gnn:')
player_df[['rusher_name','plays','mean_actual','mean_expected','mean_gravity','gravity_z','gravity_pct']].head(15)

## Next steps (v2)
- Add stunt detection so screeners and loopers get separate expected-attention conditioning.
- Switch to a temporal GNN (e.g. EvolveGCN or a small Transformer over frames) once the static frame-graph baseline is stable.
- Add edge-type heterogeneity (blocker→rusher candidate vs same-team) instead of relying on edge feature flags.
- Replace the cap at 3.0 s with the actual TTT from the play if available, to avoid truncating long dropbacks.